In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/best/default/checkpoints/epoch=97-step=20972.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader()
data_test = get_data(loader_test)
data_test.shape

Pick 2 masks, then perform linear interpolation.

In [ ]:
from utils.utils import show_samples

x1 = data_test[1].unsqueeze(0)
x2 = data_test[19].unsqueeze(0)

# Show masks
x = torch.cat([x1, x2], dim=0)
samples = torch.argmax(x, dim=1).unsqueeze(1)
show_samples(samples, rgb=False, ncol=2, figsize=(2, 4))

In [ ]:
with torch.no_grad():
    model.eval()
    model.to(device)
    x1 = x1.to(device)
    x2 = x2.to(device)

    zs1 = model.get_latent(x1, test=True)
    zs2 = model.get_latent(x2, test=True)

In [ ]:
from arch.nvae.decoder import DecoderCombinerCell

def lerp_between_latents(
    model: NVAE,
    zs1: list[torch.Tensor],
    zs2: list[torch.Tensor],
    num_interpolations: int = 10,
):
    """
    Linearly interpolate between the latent representations of two samples,
    then visualise the reconstructions.
    
    noqa: Only single samples are supported.
    """
    assert zs1[0].shape[0] == 1
    assert zs2[0].shape[0] == 1
    
    zs = []
    
    for z1, z2 in zip(zs1, zs2):
        z_buffer = []
        
        for i in range(num_interpolations):
            z = torch.lerp(z1, z2, i / (num_interpolations - 1))
            z_buffer.append(z)
        
        z_buffer = torch.stack(z_buffer).squeeze(1)
        zs.append(z_buffer)
    
    idx_dec = 0
    
    z = zs[idx_dec]
    
    # [1, top_channels, width, height]
    x = model.decoder.top_prior.unsqueeze(0)
    # [num_samples, top_channels, width, height]
    x = x.expand(num_interpolations, -1, -1, -1)
    
    for cell in model.decoder.tower:
        if isinstance(cell, DecoderCombinerCell):
            if idx_dec > 0:
                z = zs[idx_dec]

            x = cell(x, z)
            
            idx_dec += 1
        else:
            x = cell(x)
    
    x_hat_logits = model.decoder.postprocess(x)
    feats_hat_logits = model.conditional_coder(x_hat_logits)
    
    return feats_hat_logits

num_interpolations = 12

samples = lerp_between_latents(model, zs1, zs2, num_interpolations)
samples = torch.argmax(samples, dim=1).unsqueeze(1)
show_samples(
    samples,
    rgb=False,
    ncol=num_interpolations,
    figsize=(4 * num_interpolations, 4),
)

Let's do linear interpolation for some more masks.

In [ ]:
loader_test = data_module.test_dataloader(shuffle=True)
data_test = get_data(loader_test)

num_pairs = 6

xs1 = data_test[:num_pairs]
xs2 = data_test[num_pairs:2 * num_pairs]

In [ ]:
samples_all = []

with torch.no_grad():
    model.eval()
    model.to(device)
    xs1 = xs1.to(device)
    xs2 = xs2.to(device)
    
    for i in range(num_pairs):
        x1 = xs1[i].unsqueeze(0)
        x2 = xs2[i].unsqueeze(0)
        
        zs1 = model.get_latent(x1, test=True)
        zs2 = model.get_latent(x2, test=True)
        
        samples = lerp_between_latents(model, zs1, zs2, num_interpolations)
        samples = torch.argmax(samples, dim=1).unsqueeze(1)
        samples_all.append(samples)

samples_all = torch.cat(samples_all, dim=0)
show_samples(
    samples_all,
    rgb=False,
    ncol=num_interpolations,
    figsize=(4 * num_interpolations, 4 * num_pairs),
)